# Chạy Thử và So Sánh Baseline Models OOP

Notebook này được tạo ra để import trực tiếp các module từ `src/` nhằm kiểm tra xem cấu trúc OOP (TFIDFRecommender, BM25Recommender) hoạt động có chính xác như phiên bản Notebook cũ không.

Bằng cách này, bạn có thể dễ dàng test nhanh model trong lúc code, hoặc đánh giá kết quả trả về của các model với nhau.

In [ ]:
# !pip install rank_bm25


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import sys
from pathlib import Path

# Thêm root folder của ai-service vào sys.path để import được src
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
from src.data.data_loader import MovieDataLoader
from src.core.tokenizer import get_tokenizer
from src.models.tfidf_search import TFIDFRecommender
from src.models.bm25_search import BM25Recommender


## 1. Load Data bằng DataLoader
Mô phỏng lại luồng load data đã được viết trong `data_loader.py`

In [13]:
# Đường dẫn tới file dataset đã làm sạch của bạn (ưu tiên parquet hoặc csv)
data_path = project_root / 'data' / 'artifacts' / 'lexical_search' / 'catalog_vi.parquet'
if not data_path.exists():
    # Thử fall back sang file csv
    data_path = project_root / 'data' / 'tmdb' / 'movies_data_fully_translated.csv'

loader = MovieDataLoader(str(data_path))
df = loader.load_catalog(language_mode='vi')
corpus, movie_ids = loader.get_corpus_and_ids(df, id_col='movieId')

print(f"Đã load {len(corpus)} movies vào corpus.")

Đã load 9811 movies vào corpus.


## 2. Khởi tạo và Train TF-IDF Model

In [14]:
tokenizer = get_tokenizer('vi')

# Khởi tạo TF-IDF với tối đa 15000 features
tfidf_model = TFIDFRecommender(tokenizer=tokenizer, max_features=15000)

print("Đang train TF-IDF...")
tfidf_model.fit(corpus, movie_ids)
print("Hoàn tất!")

c:\Users\THUYEN\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang train TF-IDF...


c:\Users\THUYEN\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Hoàn tất!


## 3. Khởi tạo và Train BM25 Model

In [15]:
bm25_model = BM25Recommender(tokenizer=tokenizer)

print("Đang train BM25...")
bm25_model.fit(corpus, movie_ids)
print("Hoàn tất!")

Đang train BM25...
Hoàn tất!


## 4. Test So Sánh Kết Quả
Giờ ta sẽ test 1 câu truy vấn xem 2 model trả về kết quả ra sao.

In [17]:
query = "người máy hủy diệt"
top_k = 5

print(f"=== KẾT QUẢ CHO QUERY: '{query}' ===\n")

# 4.1. TF-IDF
tfidf_results = tfidf_model.search(query, top_k=top_k)
print("--- TF-IDF Top 5 ---")
for idx, res in enumerate(tfidf_results):
    m_id = res['movie_id']
    movie_title = df[df['movieId'] == m_id]['title_vi'].values[0] if 'title_vi' in df.columns else df[df['movieId'] == m_id]['original_title'].values[0]
    movie_overview = df[df['movieId'] == m_id]['overview_vi'].values[0] if 'overview_vi' in df.columns else df[df['movieId'] == m_id]['overview'].values[0]
    print(f"{idx+1}. ID: {m_id: <6} | Score: {res['score']:.4f} | Title: {movie_title} | Overview: {movie_overview}")

print("\n")

# 4.2. BM25
bm25_results = bm25_model.search(query, top_k=top_k)
print("--- BM25 Top 5 ---")
for idx, res in enumerate(bm25_results):
    m_id = res['movie_id']
    movie_title = df[df['movieId'] == m_id]['title_vi'].values[0] if 'title_vi' in df.columns else df[df['movieId'] == m_id]['original_title'].values[0]
    movie_overview = df[df['movieId'] == m_id]['overview_vi'].values[0] if 'overview_vi' in df.columns else df[df['movieId'] == m_id]['overview'].values[0]
    print(f"{idx+1}. ID: {m_id: <6} | Score: {res['score']:.4f} | Title: {movie_title} | Overview: {movie_overview}")

=== KẾT QUẢ CHO QUERY: 'người máy hủy diệt' ===

--- TF-IDF Top 5 ---
1. ID: 68791  | Score: 0.3087 | Title: Kẻ Hủy Diệt 4 | Overview: “Terminator Salvation” là bộ phim thứ tư của dòng phim hành động nổi tiếng “Terminator” được phát hành phần đầu vào năm 1984. Bối cảnh phim diễn ra trong thế giới giả tưởng của tương lai năm 2018, khi cuộc chiến của nhân loại với những cỗ máy hủy diệt do hệ thống máy tính thông minh siêu việt tên Skynet làm chủ. Sau khi âm mưu xóa sổ nhân loại trong hàng trăm vụ nổ hạt nhân, Skynet và binh đoàn hủy diệt của nó bắt đầu tiến hành chiến dịch truy quét và tàn sát những người sống sót.
2. ID: 108508 | Score: 0.2577 | Title: Mother of George | Overview: Một cặp vợ chồng đang sống ở Brooklyn gặp khó khăn trong việc nuôi con - một vấn đề bất lợi cho nền văn hóa và dẫn đến một quyết định sốc có thể cứu hoặc hủy diệt chúng.
3. ID: 6537   | Score: 0.2549 | Title: Kẻ Hủy Diệt 3: Người Máy Nổi Loạn | Overview: Đã một thập kỷ kể từ khi John Connor - Nick Stahl - cứu 